# Decision threshold, calibration, and model explanations

This notebook turns the selected Temperature + Light + CO₂ logistic model's probabilities into explicit occupancy decisions. It selects thresholds using chronological training validation only, checks whether probabilities behave like probabilities, and explains global behavior and representative errors.

## 1. Load packaged Phase 6 results

Run `python -m sensorbudget.modeling.decision_explainability` before opening this notebook. The notebook does not retrain or select models.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULT_DIR = PROJECT_ROOT / "models" / "decision_explainability"
files = {
    "curves": "threshold_curves.csv",
    "selected": "selected_thresholds.csv",
    "heldout": "heldout_metrics.csv",
    "calibration": "calibration_bins.csv",
    "calibration_summary": "calibration_summary.csv",
    "coefficients": "global_coefficients.csv",
    "local": "representative_explanations.csv",
    "transition": "transition_metrics.csv",
}
missing = [name for name in files.values() if not (RESULT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing Phase 6 artifacts: {missing}")
tables = {name: pd.read_csv(RESULT_DIR / path) for name, path in files.items()}
curves = tables["curves"]
selected = tables["selected"]
heldout = tables["heldout"]
calibration = tables["calibration"]
calibration_summary = tables["calibration_summary"]
coefficients = tables["coefficients"]
local = tables["local"]
transition = tables["transition"]
PLOTLY_TEMPLATE = "plotly_white"
SPLIT_LABELS = {"test_1": "Test 1", "test_2": "Test 2"}
print(f"Loaded {len(curves)} threshold evaluations.")

## 2. Error-cost assumptions

A false occupied decision may waste energy by conditioning an empty room. A false unoccupied decision may reduce comfort or ventilation while people are present. Because the dataset cannot tell us their monetary value, three transparent assumptions are compared.

In [ ]:
cost_table = selected[[
    "scenario_label", "false_positive_cost", "false_negative_cost"
]].drop_duplicates().rename(columns={
    "scenario_label": "Scenario",
    "false_positive_cost": "False occupied cost",
    "false_negative_cost": "False unoccupied cost",
})
display(cost_table)

**Interpretation.** Equal cost is the reference operating assumption. Comfort-focused makes a missed occupied row five times as costly; energy-focused makes an unnecessary occupied response five times as costly. These are sensitivity assumptions, not measured building economics.

## 3. Threshold selection on chronological validation

Every point applies one classification threshold to out-of-fold training probabilities. Lower is better. Test 1 and Test 2 do not appear in this calculation.

In [ ]:
fig = go.Figure()
colors = {
    "Equal error cost": "#457b9d",
    "Comfort-focused": "#2a9d8f",
    "Energy-focused": "#e76f51",
}
for label in ["Equal error cost", "Comfort-focused", "Energy-focused"]:
    rows = curves.loc[curves["scenario_label"] == label]
    chosen = selected.loc[selected["scenario_label"] == label].iloc[0]
    fig.add_scatter(
        x=rows["threshold"], y=rows["cost_per_1000_rows"],
        mode="lines", name=label, line={"color": colors[label]},
        hovertemplate=(
            f"{label}<br>Threshold: %{{x:.2f}}"
            "<br>Cost per 1,000 rows: %{y:.2f}<extra></extra>"
        ),
    )
    fig.add_scatter(
        x=[chosen["threshold"]], y=[chosen["cost_per_1000_rows"]],
        mode="markers", marker={"size": 10, "color": colors[label]},
        name=f"{label} selected", showlegend=False,
        hovertemplate=(
            f"{label} selected<br>Threshold: %{{x:.2f}}"
            "<br>Cost per 1,000 rows: %{y:.2f}<extra></extra>"
        ),
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE, title="Validation error cost across thresholds",
    xaxis_title="Classification threshold",
    yaxis_title="Assumed error cost per 1,000 validation rows",
    height=500,
)
fig.show()

In [ ]:
selection_table = selected[[
    "scenario_label", "threshold", "cost_per_1000_rows",
    "default_cost_per_1000_rows", "precision", "recall", "f1",
]].rename(columns={
    "scenario_label": "Scenario", "threshold": "Selected threshold",
    "cost_per_1000_rows": "Selected cost/1,000",
    "default_cost_per_1000_rows": "Cost/1,000 at 0.5",
    "precision": "Precision", "recall": "Recall", "f1": "F1",
})
display(selection_table.style.format({
    "Selected threshold": "{:.2f}", "Selected cost/1,000": "{:.2f}",
    "Cost/1,000 at 0.5": "{:.2f}", "Precision": "{:.3f}",
    "Recall": "{:.3f}", "F1": "{:.3f}",
}))

**Conclusion.** Equal-cost validation selects 0.86, reducing assumed validation cost from 11.05 to 5.75 per 1,000 rows. Comfort-focused also selects 0.86, while energy-focused selects 0.88. The probabilities are highly separated on validation, so a high threshold removes false positives without initially losing many occupied rows.

## 4. Does the selected threshold generalize?

The equal-cost threshold is now frozen. The chart compares it with 0.5 on the two held-out source periods, whose labels were not used to select the threshold. The comparison diagnoses stability; it does not retroactively retune the threshold or create fresh confirmation data.

In [ ]:
comparison = heldout.loc[heldout["scenario"] == "equal_cost"].copy()
comparison["split_label"] = comparison["split"].map(SPLIT_LABELS)
source_labels = {
    "validation_selected": "Validation-selected (0.86)",
    "default_0.5": "Default (0.50)",
}
comparison["threshold_label"] = comparison["threshold_source"].map(source_labels)
fig = go.Figure()
for threshold_source, color in [
    ("default_0.5", "#a8dadc"),
    ("validation_selected", "#457b9d"),
]:
    rows = comparison.loc[comparison["threshold_source"] == threshold_source]
    fig.add_bar(
        x=rows["cost_per_1000_rows"], y=rows["split_label"],
        orientation="h", name=source_labels[threshold_source],
        marker_color=color,
        text=rows["cost_per_1000_rows"].map(lambda value: f"{value:.2f}"),
        textposition="outside",
        hovertemplate=(
            "%{y}<br>Cost per 1,000 rows: %{x:.2f}<extra></extra>"
        ),
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE, barmode="group",
    title="Equal-cost held-out comparison: selected versus default threshold",
    xaxis_title="Assumed error cost per 1,000 rows", yaxis_title=None,
    legend={"traceorder": "reversed"}, height=420,
)
fig.show()

**Conclusion.** The validation advantage does not transfer. On Test 1, cost is 22.51 at 0.86 versus 21.39 at 0.5. On Test 2, it rises from 8.41 to 22.05 because recall falls from 0.994 to 0.918. We keep 0.86 as the auditable validation-selected reference, but the temporal instability prevents a deployment recommendation.

## 5. Probability calibration

A calibrated probability of 0.8 should correspond to occupancy roughly 80% of the time. The dotted diagonal represents perfect calibration. Point size shows how many rows occupy each probability bin.

In [ ]:
dataset_labels = {
    "chronological_cv": "Chronological CV",
    "test_1": "Test 1", "test_2": "Test 2",
}
fig = go.Figure()
fig.add_scatter(
    x=[0, 1], y=[0, 1], mode="lines", name="Perfect calibration",
    line={"dash": "dot", "color": "#777777"},
)
for dataset, color in [
    ("chronological_cv", "#457b9d"),
    ("test_1", "#2a9d8f"), ("test_2", "#e76f51"),
]:
    rows = calibration.loc[
        (calibration["dataset"] == dataset) & (calibration["row_count"] > 0)
    ]
    fig.add_scatter(
        x=rows["mean_predicted_probability"],
        y=rows["observed_occupancy_rate"], mode="lines+markers",
        name=dataset_labels[dataset], line={"color": color},
        marker={"size": 7 + rows["row_count"] ** 0.35},
        customdata=rows[["row_count"]],
        hovertemplate=(
            "Mean predicted: %{x:.3f}<br>Observed: %{y:.3f}"
            "<br>Rows: %{customdata[0]:.0f}<extra></extra>"
        ),
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE, title="Reliability diagram",
    xaxis={"title": "Mean predicted occupancy probability", "range": [0, 1.01]},
    yaxis={"title": "Observed occupancy rate", "range": [0, 1.01]},
    height=520,
)
fig.show()
display(calibration_summary.round(4))

**Conclusion.** Aggregate Brier scores are low, but reliability differs across periods. Expected calibration error is 0.027 in validation, 0.012 on Test 1, and 0.022 on Test 2. No calibration transform is fitted because doing so credibly would require another nested temporal validation comparison.

## 6. Global model behavior

Because the inputs were standardized before logistic regression, these coefficients compare the effect of increasing each feature by one training standard deviation while holding the others fixed. They describe model associations, not causal effects.

In [ ]:
coefficient_rows = coefficients.sort_values("standardized_coefficient")
fig = go.Figure(go.Bar(
    x=coefficient_rows["standardized_coefficient"],
    y=coefficient_rows["feature"], orientation="h",
    marker_color=[
        "#2a9d8f" if value >= 0 else "#e76f51"
        for value in coefficient_rows["standardized_coefficient"]
    ],
    text=coefficient_rows["standardized_coefficient"].map(
        lambda value: f"{value:+.2f}"
    ), textposition="outside",
    customdata=list(zip(
        coefficient_rows["standardized_coefficient"].map(
            lambda value: f"{value:+.3f}"
        ),
        coefficient_rows["odds_ratio_per_training_sd"].map(
            lambda value: f"{value:.2f}"
        ),
    )),
    hovertemplate=(
        "%{y}<br>Standardized coefficient: %{customdata[0]}"
        "<br>Odds ratio per training SD: %{customdata[1]}<extra></extra>"
    ),
))
fig.add_vline(x=0, line_color="#555555")
fig.update_layout(
    template=PLOTLY_TEMPLATE, title="Standardized logistic-regression coefficients",
    xaxis_title="Change in occupancy log-odds per one training SD",
    yaxis_title=None, height=400,
)
fig.show()

**Conclusion.** Light has the largest positive coefficient (+4.50), followed by CO₂ (+1.86). Temperature has a negative conditional coefficient (-1.23). This confirms that the selected model remains strongly dependent on the Light proxy documented in earlier phases.

## 7. Representative errors and local contributions

The table shows the false positive and false negative closest to the 0.86 boundary in each test period. A positive contribution pushes toward occupied; a negative contribution pushes toward unoccupied. Contributions plus the intercept reconstruct the logistic score.

In [ ]:
errors = (
    local.loc[local["outcome"].isin(["false_positive", "false_negative"])]
    .sort_values("distance_from_threshold")
    .groupby(["source_split", "outcome"], as_index=False, group_keys=False)
    .head(1)
    .copy()
)
errors["case"] = (
    errors["source_split"].map(SPLIT_LABELS) + " — "
    + errors["outcome"].str.replace("_", " " )
)
display(errors[[
    "case", "date", "probability_occupied", "Temperature",
    "Light", "CO2", "Temperature_contribution",
    "Light_contribution", "CO2_contribution",
]].round(3))
fig = go.Figure()
for feature, color in [
    ("Temperature", "#e76f51"), ("Light", "#f4a261"),
    ("CO2", "#2a9d8f"),
]:
    fig.add_bar(
        x=errors[f"{feature}_contribution"], y=errors["case"],
        orientation="h", name=feature,
        customdata=errors[f"{feature}_contribution"].map(
            lambda value: f"{value:+.3f}"
        ),
        hovertemplate=(
            f"%{{y}}<br>{feature} contribution: "
            "%{customdata}<extra></extra>"
        ),
        marker_color=color,
    )
fig.add_vline(x=0, line_color="#555555")
fig.update_layout(
    template=PLOTLY_TEMPLATE, barmode="relative",
    title="Feature contributions for boundary-near held-out errors",
    xaxis_title="Contribution to occupancy log-odds", yaxis_title=None,
    height=480, margin={"l": 180},
)
fig.show()

**Conclusion.** Individual decisions are additive and auditable, but they inherit the same Light dominance as the global model. A local contribution explains why this model produced a score; it does not prove the feature caused occupancy.

## 8. False negatives after occupancy starts

Recall is grouped by minutes since the most recent transition into occupancy. This distinguishes short detection delay from errors that persist during established occupancy.

In [ ]:
transition["split_label"] = transition["split"].map(SPLIT_LABELS)
phase_order = ["0–2 min", "3–5 min", "6–15 min", ">15 min"]
fig = go.Figure()
for split, color in [("Test 2", "#e76f51"), ("Test 1", "#457b9d")]:
    rows = transition.loc[transition["split_label"] == split].copy()
    rows["occupancy_phase"] = pd.Categorical(
        rows["occupancy_phase"], categories=phase_order, ordered=True
    )
    rows = rows.sort_values("occupancy_phase", ascending=False)
    fig.add_bar(
        x=rows["recall"], y=rows["occupancy_phase"], orientation="h",
        name=split, marker_color=color,
        text=rows["recall"].map(lambda value: f"{value:.3f}"),
        textposition="outside", customdata=rows[["false_negative"]],
        hovertemplate=(
            "%{y}<br>Recall: %{x:.3f}"
            "<br>False negatives: %{customdata[0]:.0f}<extra></extra>"
        ),
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE, barmode="group",
    title="Recall by time since occupancy onset at threshold 0.86",
    xaxis={"title": "Recall", "range": [0, 1.01]}, yaxis_title=None,
    legend={"traceorder": "reversed"}, height=480,
)
fig.show()

**Conclusion.** Test 1 shows a short onset effect and then nearly perfect recall. Test 2 recall is 0.857 in the first 0–2 minutes but remains only 0.918 after 15 minutes. Its false negatives therefore reflect broader period shift, not only delayed response at occupancy transitions.

## 9. Phase 6 conclusion

The original model used the conventional threshold of **0.50** and achieved excellent held-out performance: F1 was 0.971 on Test 1 and 0.980 on Test 2. Phase 6 then asked whether a different threshold could reduce an explicitly defined operational error cost.

Under the equal-error-cost assumption, chronological validation selected **0.86**. Within validation, this reduced assumed error cost from 11.05 to 5.75 units per 1,000 rows while retaining high precision and recall. This was a legitimate, leakage-safe result because Test 1 and Test 2 did not influence the selection.

However, the expected benefit did not generalize. On Test 1, 0.86 was slightly worse than 0.50. On Test 2, equal-error cost increased from 8.41 to 22.05 units per 1,000 rows, F1 fell from 0.980 to 0.946, and recall fell from 0.994 to 0.918. Many genuinely occupied Test 2 rows received probabilities between 0.50 and 0.86 and therefore became false negatives at the higher threshold.

The appropriate conclusion is therefore **not** that 0.86 is more cost-effective in general. It was more cost-effective only in the available validation periods. Test 2 reveals that the probability distribution and cost-optimal threshold are not temporally stable. We retain 0.86 in the report as the auditable validation-selected result, but we do not recommend deploying it.

The original 0.50 threshold remains the stronger observed held-out reference, but it should not be re-declared optimal from the test results because the held-out periods are not tuning data. A real operating point requires stakeholder-owned energy and comfort costs, broader validation across buildings and seasons, and prospective monitoring that can detect probability or threshold drift. The coefficient and local-contribution analysis adds a related limitation: the model remains strongly dependent on Light, so changes in lighting behavior can alter both probabilities and the appropriate threshold.